In [ ]:
from tokenise import BPE_token

import os
import gc
import pandas as pd
from os.path import join
import numpy as np
import torch
import torch.nn as nn
from tqdm.notebook import tqdm
from transformers import GPT2Config, GPT2LMHeadModel, GPT2Tokenizer

from sklearn.metrics import  accuracy_score
from transformers import (
                          GPT2Config,
                          GPT2Tokenizer,
                          get_linear_schedule_with_warmup,)

from scipy.spatial.distance import cosine
from collections import OrderedDict
from math import ceil
import torch.nn.functional as F
from sklearn.model_selection import train_test_split
from torch.optim import AdamW


In [ ]:
if torch.cuda.is_available() :
    device = torch.device("cuda")

In [ ]:
tokenizer = BPE_token()
save_path = './tokenized_data'
# Load the tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(save_path)
tokenizer.add_special_tokens({
    "eos_token": "</s>",
    "bos_token": "<s>",
    "unk_token": "<unk>",
    "pad_token": "</s>",###</s>
    "mask_token": "<mask>"
})
tokenizer.padding_side='left' # because it's gpt

In [ ]:
layers = 2
#per_for_train = '10'
per_train_list = ['_human']
temp=1
lr = 5e-6
labels_ids = {'true': 0, 'false': 1}
n_labels = len(labels_ids)
max_length = 25
epochs = 100
batch_size = 20

read_file = pd.read_excel('./fmri_map_id.xlsx')

subjects = list(read_file['BIDS_ID'])

In [ ]:
from right_corpus import NP, NPVP, SOV
print(len(NP + NPVP + SOV))

In [ ]:

df_ma = pd.read_excel('./MATERIAL.xlsx')
id_2_dur=dict(zip(df_ma['TrialID'],df_ma['soundDur']))


trail_list = [ str(df_ma['word1'][i]) + ' ' + str(df_ma['word2'][i]) + ' '  + str(df_ma['word3'][i]) + ' ' + str(df_ma['word4'][i]) + ' '   
              + str(df_ma['word5'][i]) + ' '   + str(df_ma['word6'][i]) + ' '   + str(df_ma['word7'][i]) + ' '   + str(df_ma['word8'][i])
               for i in range(len(df_ma))]
trail_list = [sentence.replace("nan", "").strip() for sentence in trail_list]


def remove_duplicates(sentence):
    words = sentence.split()  
    for i in range(7):
        if words[-1] == '#':
            words.pop(-1)
        if words[-1] ==words[-2]:
            words.pop(-1)
    return ' '.join(words) 


new_list_raw = [remove_duplicates(sentence) for sentence in trail_list]

y_train_raw = []
for itemt in new_list_raw:
    if itemt in NP + NPVP + SOV:
        y_train_raw.append('true')
    else:
        y_train_raw.append('false')

id_2_trail_type = dict(zip(list(df_ma['TrialID']),new_list_raw))


mapping = {}
new_list_co = []
new_list_unc = []
for n_num,co in enumerate(new_list_raw):
    if co in (NP + NPVP + SOV):
        new_list_co.append(co)
        mapping[n_num] = len(new_list_co) - 1  
    else:
        new_list_unc.append(co)
        mapping[n_num] = 144 + len(new_list_unc) - 1

new_list = new_list_co + new_list_unc

y_train = []
for itemt in new_list:
    if itemt in (NP + NPVP + SOV):
        y_train.append('true')
    else:
        y_train.append('false')

y_train_tensor = []
for itemt in new_list:
    if itemt in NP + NPVP + SOV:
        y_train_tensor.append(torch.tensor(0).to(device))
    else:
        y_train_tensor.append(torch.tensor(1).to(device))

In [ ]:
#requires restricted data available upon request
def first_day_test(subject):   
    id_2_beh_list = []
    for blk_num in range(1,3):
        df_sub_1 =  pd.read_csv(f'./output_csv/BCT_fMRI_GJT_blk{blk_num}-{subject}-1.csv')
        train_id_list = list(map(int, df_sub_1['TrialID'].dropna()))
        sub_beh = list(map(int, df_sub_1['Stimuli3.ACC'].dropna()))
        id_2_beh = dict(zip(train_id_list,sub_beh))
        id_2_beh_list.append(id_2_beh)
    for blk_num in range(3,5):
        df_sub_1 =  pd.read_csv(f'./output_csv/BCT_fMRI_GJT_blk{blk_num}-{subject}-1.csv')
        train_id_list = list(map(int, df_sub_1['TrialID'].dropna()))
        sub_beh = list(map(int, df_sub_1['Stimuli3.ACC'].dropna()))
        train_id_list = [(i+96) for i in train_id_list ]
        id_2_beh = dict(zip(train_id_list,sub_beh))
        id_2_beh_list.append(id_2_beh)
    for blk_num in range(5,7):
        df_sub_1 =  pd.read_csv(f'./output_csv/BCT_fMRI_GJT_blk{blk_num}-{subject}-1.csv')
        train_id_list = list(map(int, df_sub_1['TrialID'].dropna()))
        train_id_list = [(i+192) for i in train_id_list ]
        sub_beh = list(map(int, df_sub_1['Stimuli3.ACC'].dropna()))
        id_2_beh = dict(zip(train_id_list,sub_beh))
        id_2_beh_list.append(id_2_beh)
    merged_id_2_beh = {}
    for id_2_beh in id_2_beh_list:
        merged_id_2_beh.update(id_2_beh)

    sorted_merged_id_2_beh = OrderedDict(sorted(merged_id_2_beh.items()))
    y_train_sub_score_raw = []
    for train_id, behavior in sorted_merged_id_2_beh.items():
        if behavior == 1:
            y_train_sub_score_raw.append(y_train_raw[(train_id-1)])
        else:
            if y_train_raw[(train_id-1)] == 'true':
                y_train_sub_score_raw.append('false')
            if y_train_raw[(train_id-1)] == 'false':
                y_train_sub_score_raw.append('true')

    y_train_sub_score = [None] * len(new_list) 


    for raw_index, new_index in mapping.items():
        y_train_sub_score[new_index] = y_train_sub_score_raw[raw_index]
    if len(set(y_train_sub_score)) != 2:
        print('!')
    return(y_train_sub_score)


In [ ]:
def second_day_test(subject):   
    id_2_beh_list = []
    for blk_num in range(1,3):
        df_sub_1 =  pd.read_csv(f'./output_csv/BCT_GJT_blk{blk_num}-{subject}-2.csv')
        train_id_list = list(map(int, df_sub_1['TrialID'].dropna()))
        sub_beh = list(map(int, df_sub_1['Stimuli3.ACC'].dropna()))
        id_2_beh = dict(zip(train_id_list,sub_beh))
        id_2_beh_list.append(id_2_beh)
    for blk_num in range(3,5):
        df_sub_1 =  pd.read_csv(f'./output_csv/BCT_GJT_blk{blk_num}-{subject}-2.csv')
        train_id_list = list(map(int, df_sub_1['TrialID'].dropna()))
        sub_beh = list(map(int, df_sub_1['Stimuli3.ACC'].dropna()))
        train_id_list = [(i+96) for i in train_id_list ]
        id_2_beh = dict(zip(train_id_list,sub_beh))
        id_2_beh_list.append(id_2_beh)
    for blk_num in range(5,7):
        df_sub_1 =  pd.read_csv(f'./output_csv/BCT_GJT_blk{blk_num}-{subject}-2.csv')
        train_id_list = list(map(int, df_sub_1['TrialID'].dropna()))
        train_id_list = [(i+192) for i in train_id_list ]
        sub_beh = list(map(int, df_sub_1['Stimuli3.ACC'].dropna()))
        id_2_beh = dict(zip(train_id_list,sub_beh))
        id_2_beh_list.append(id_2_beh)
    merged_id_2_beh = {}
    for id_2_beh in id_2_beh_list:
        merged_id_2_beh.update(id_2_beh)

    sorted_merged_id_2_beh = OrderedDict(sorted(merged_id_2_beh.items()))
    y_train_sub_score_raw = []
    for train_id, behavior in sorted_merged_id_2_beh.items():
        if behavior == 1:
            y_train_sub_score_raw.append(y_train_raw[(train_id-1)])
        else:
            if y_train_raw[(train_id-1)] == 'true':
                y_train_sub_score_raw.append('false')
            if y_train_raw[(train_id-1)] == 'false':
                y_train_sub_score_raw.append('true')

    y_train_sub_score = [None] * len(new_list)  

    for raw_index, new_index in mapping.items():
        y_train_sub_score[new_index] = y_train_sub_score_raw[raw_index]
    if len(set(y_train_sub_score)) != 2:
        print('!')
    return(y_train_sub_score)



In [ ]:
id_item = 23001

df_subject = pd.read_csv(f'./output_csv/GJT_generalization-{id_item}-7.csv')
#id_2_dur=dict(zip(df_ma['TrialID'],df_ma['soundDur']))


trail_list = [ str(df_subject['word1'][i]) + ' ' + str(df_subject['word2'][i]) + ' '  + str(df_subject['word3'][i]) + ' ' + str(df_subject['word4'][i]) + ' '   
            + str(df_subject['word5'][i]) + ' '   + str(df_subject['word6'][i]) + ' '   + str(df_subject['word7'][i]) + ' '   + str(df_subject['word8'][i])
            for i in range(len(df_subject))]
trail_list = [sentence.replace("nan", "").strip() for sentence in trail_list]


def remove_duplicates(sentence):
    words = sentence.split()  
    if sentence:
        for i in range(7):
            if words[-1] == '#':
                words.pop(-1)
            if words[-1] ==words[-2]:
                words.pop(-1)
    return ' '.join(words)  


sub_item_list = [remove_duplicates(sentence) for sentence in trail_list]
used_sub_item_list = [sentence for sentence in sub_item_list if sentence]


result1 = df_subject['Stimuli2.ACC'].astype(float)
result1_resp = df_subject['Stimuli2.CRESP']
result1 = result1.dropna()
result1_resp = result1_resp.dropna()
result_int1 = result1.astype(int)

result2 = df_subject['Stim2.ACC'].astype(float)
result2_resp = df_subject['Stim2.CRESP']
result2 = result2.dropna()
result2_resp = result2_resp.dropna()
result_int2 = result2.astype(int)

result3 = df_subject['StimC2.ACC'].astype(float)
result3_resp = df_subject['StimC2.CRESP']
result3 = result3.dropna()
result3_resp = result3_resp.dropna()
result_int3 = result3.astype(int)

sub_item_train = [] 
for idx, sub_acc in enumerate((list(result1_resp) + list(result2_resp) + list(result3_resp))):
    if sub_acc == 'j':
        sub_item_train.append('true')
    else:
        sub_item_train.append('false')

base_used_sub_item_list = used_sub_item_list
base_label = sub_item_train


In [ ]:
def seven_day_gen_test(id_item):

    df_subject = pd.read_csv(f'./output_csv/GJT_generalization-{id_item}-7.csv')



    trail_list = [ str(df_subject['word1'][i]) + ' ' + str(df_subject['word2'][i]) + ' '  + str(df_subject['word3'][i]) + ' ' + str(df_subject['word4'][i]) + ' '   
                + str(df_subject['word5'][i]) + ' '   + str(df_subject['word6'][i]) + ' '   + str(df_subject['word7'][i]) + ' '   + str(df_subject['word8'][i])
                for i in range(len(df_subject))]
    trail_list = [sentence.replace("nan", "").strip() for sentence in trail_list]


    def remove_duplicates(sentence):
        words = sentence.split()  
        if sentence:
            for i in range(7):
                if words[-1] == '#':
                    words.pop(-1)
                if words[-1] ==words[-2]:
                    words.pop(-1)
        return ' '.join(words)  

    sub_item_list = [remove_duplicates(sentence) for sentence in trail_list]
    used_sub_item_list = [sentence for sentence in sub_item_list if sentence]

    
    sub_item_train = []
    for item in used_sub_item_list:
        label = base_label[base_used_sub_item_list.index(item)]
        sub_item_train.append(label)

    result1 = df_subject['Stimuli2.ACC'].astype(float)
    result1 = result1.dropna()
    result_int1 = result1.astype(int)

    result2 = df_subject['Stim2.ACC'].astype(float)
    result2 = result2.dropna()
    result_int2 = result2.astype(int)

    result3 = df_subject['StimC2.ACC'].astype(float)
    result3 = result3.dropna()
    result_int3 = result3.astype(int)

    sub_seven_acc_zero_one = []
    for idx, sub_acc in enumerate((list(result_int1) + list(result_int2) + list(result_int3))):
        if sub_acc == 1:
            sub_seven_acc_zero_one.append(sub_item_train[idx])
        else:
            if sub_item_train[idx] == 'true':
                sub_seven_acc_zero_one.append('false')
            if sub_item_train[idx] == 'false':
                sub_seven_acc_zero_one.append('true')
            
    return(used_sub_item_list, sub_item_train, sub_seven_acc_zero_one)

In [ ]:
def tf_to01_list(xs):
    out = []
    for x in xs:
        if isinstance(x, str):
            s = x.strip().lower()
            if s == 'true':
                out.append(0)
            elif s == 'false':
                out.append(1)
            elif s in ('0', '1'):
                out.append(int(s))
            else:
                raise ValueError(f"Unexpected token: {x}")
        elif isinstance(x, bool):
            out.append(0 if x else 1)
        elif x in (0, 1):
            out.append(int(x))
        else:
            raise ValueError(f"Unexpected type/value: {x}")
    return np.asarray(out, dtype=np.int8)


In [ ]:

all_brain_data_vec = []

for subject in subjects:
    all_brain_data_vec.append(np.load(f'./gpt_f_masked_neural_data/sub-{subject}_neural_vector.npy'))
    

print(len(all_brain_data_vec))

In [ ]:
acc_array_list = np.zeros((102,100,288))
similarity_array_list = np.zeros((102,100,288))
present_array_list = np.zeros((102,100,288,768))
train_acc_list = np.zeros((102,5,100))
valid_acc_list = np.zeros((102,5,100))
test_acc_list =  np.zeros((102,5,100))
like_human_acc_list =  np.zeros((102,5,100))
np_acc_list =  np.zeros((102,5,100))
npvp_acc_list =  np.zeros((102,5,100))
sov_acc_list =  np.zeros((102,5,100))
mean_sim_list =  np.zeros((102,5,100))
criterion_cla = nn.CrossEntropyLoss()
criterion_cor = nn.MSELoss()
criterion_cos_metric = nn.CosineEmbeddingLoss(margin=0.0)
gen_result = np.zeros((102,5))
day7_predictions = [[] for _ in range(len(subjects))]
chonghe_acc_list =  np.zeros((102,5,100))
valid_acc_list_ture = np.zeros((102,5,100))

def process_subject(subject):

    
    print(f'{subject}-start')
    biggest_like_huamn = 0

    best_model_path = f'./gpt_f__mask_best_model_block/{subject}'

    os.makedirs(best_model_path, exist_ok=True)
    sub_reresult = []
    used_sub_item_list = seven_day_gen_test(subject)[0]
    sub_item_train = seven_day_gen_test(subject)[1]
    sub_seven_acc_zero_one = seven_day_gen_test(subject)[2]
    sub_seven_acc_zero_one_to01 = tf_to01_list(sub_seven_acc_zero_one)
    first_y_train = first_day_test(subject)
    first_y_train_tensor = []
    for first_label in first_y_train:
        if first_label == 'true':
            first_y_train_tensor.append(torch.tensor(0).to(device))
        else:
            first_y_train_tensor.append(torch.tensor(1).to(device))

    second_y_train = second_day_test(subject)
    second_y_train_tensor = []
    for second_label in second_y_train:
        if second_label == 'true':
            second_y_train_tensor.append(torch.tensor(0).to(device))
        else:
            second_y_train_tensor.append(torch.tensor(1).to(device))

    sub_item_train_tensor = []
    for subitemm in sub_item_train:
        if subitemm == 'true':
            sub_item_train_tensor.append(torch.tensor(0).to(device))
        else:
            sub_item_train_tensor.append(torch.tensor(1).to(device))
    print('Epoch')

    train_acc_array = np.zeros((5,100))
    valid_acc_array = np.zeros((5,100))
    test_acc_array = np.zeros((5,100))
    like_human_acc_array = np.zeros((5,100))
    np_acc_array = np.zeros((5,100))
    npvp_acc_array = np.zeros((5,100))
    sov_acc_array = np.zeros((5,100))
    mean_sim_array = np.zeros((5,100))

    acc_array = np.zeros((100,288)) #predict 0 or 1
    similarity_array = np.zeros((100,288))
    present_array = np.zeros((100,288,768))
    new_rate_array = np.zeros((5))
    sub_day7_predictions = []
    chonghe_acc_array = np.zeros((5,100))
    valid_acc_array_true = np.zeros((5,100))
    fold_idx = 0
    

    train_index, test_index = train_test_split(range(len(new_list)), test_size=0.3, random_state=42, shuffle=True)
    for idddd in range(1):
        config = GPT2Config(
            vocab_size=tokenizer.vocab_size,
            bos_token_id=tokenizer.bos_token_id,
            eos_token_id=tokenizer.eos_token_id,
            n_layer=layers ,
            )

            # Create the model and move it to GPU
        model = GPT2LMHeadModel(config).to(device)

        linear_net = nn.Linear(in_features=768, out_features=2, bias=True)
        linear_net = linear_net.to(device)
    
            # Set the attention mask and pad token id
        model.config.pad_token_id = tokenizer.eos_token_id   

        random_indices = train_index
        remaining_indices = test_index
        train_list = [new_list[i] for i in random_indices]
        #train_labels = [first_y_train_tensor[i] for i in random_indices]
        train_labels_1 = [first_y_train_tensor[i] for i in random_indices]
        train_labels_2 = [second_y_train_tensor[i] for i in random_indices]
        val_list = [new_list[i] for i in remaining_indices]

        val_labels = [y_train_tensor[i] for i in remaining_indices]

        val_labels_1 = [first_y_train_tensor[i] for i in remaining_indices]
        val_labels_2 = [second_y_train_tensor[i] for i in remaining_indices]


        input_tensor = torch.tensor(all_brain_data_vec[subjects.index(subject)], dtype=torch.float32).to(device)
        train_neural_data = [input_tensor[(i)] for i in random_indices]#permute
        test_neural_data = [input_tensor[i] for i in remaining_indices]

        optimizer = AdamW(list(model.parameters()) + list(linear_net.parameters()),
                    lr = 5e-5, # default is 5e-5, our notebook had 2e-5
                    eps = 1e-8 # default is 1e-8.
                    )
        batch_size = 32
        updates_per_epoch = ceil(len(train_list) / batch_size)
        total_steps = updates_per_epoch * epochs

        scheduler = get_linear_schedule_with_warmup(
            optimizer, num_warmup_steps=0, num_training_steps=total_steps
        )
      
        
        
        for epoch in tqdm(range(epochs), disable=True):

            loss_all = torch.zeros(1).to(device)

            lm_loss = torch.zeros(1).to(device)
            model.train()
            linear_net.train()
            for cor_id, corpus in enumerate(train_list):
                output = model(torch.tensor([tokenizer.encode(corpus)]).to(model.device),
                            labels=torch.tensor([tokenizer.encode(corpus)]).to(device),
                            output_hidden_states=True)
                model_label = linear_net(output.hidden_states[-1][-1][-1])
                if (epoch // 30 ) % 2 == 0:
                    loss_c = criterion_cla(model_label.unsqueeze(0), train_labels_2[cor_id].unsqueeze(0))  
                else:
                    loss_c = criterion_cla(model_label.unsqueeze(0), train_labels_1[cor_id].unsqueeze(0))
                
                loss_mse = criterion_cor(output.hidden_states[-1][-1][-1],train_neural_data[cor_id])
               
                loss_all += (np.log(79/2)*loss_c)+((loss_mse * 4)/768)
                if corpus in new_list_co:
                    lm_loss +=  (output.loss)
                
                if ((cor_id+1) % batch_size) == 0:
                    
                    combined_loss = (lm_loss) + loss_all
                    optimizer.zero_grad()   
                    combined_loss.backward()  
                    optimizer.step()         
                    scheduler.step()
                    loss_all = torch.zeros(1).to(device) 
                    lm_loss = torch.zeros(1).to(device)
            
            
            if len(train_list) % batch_size != 0:
                combined_loss = (lm_loss) + loss_all
                optimizer.zero_grad()
                combined_loss.backward()
                optimizer.step()
                scheduler.step()
            
            with torch.no_grad():
                model.eval()
                linear_net.eval()
                correct_predictions = 0
                correct_predictions_1 = 0
                correct_predictions_2 = 0
                total_predictions = 0
                cosine_value_list = []
                for cor_id, corpus in enumerate(val_list):
                  
                    output = model(torch.tensor([tokenizer.encode(corpus)]).to(model.device), output_hidden_states=True)
                    cosine_value = 1-cosine(output.hidden_states[-1][-1][-1].cpu(), test_neural_data[cor_id].cpu())
                    cosine_value_list.append(cosine_value)
                    similarity_array[epoch, remaining_indices[cor_id]] = cosine_value
                    present_array[epoch, remaining_indices[cor_id]] = output.hidden_states[-1][-1][-1].detach().cpu().numpy()

                    model_label = linear_net(output.hidden_states[-1][-1][-1])
                    
                    _, predicted_label = torch.max(model_label, dim=0)
                    true_label = val_labels[cor_id].item()
                    true_label_1 = val_labels_1[cor_id].item()
                    true_label_2 = val_labels_2[cor_id].item()

               
                    if predicted_label.item() == true_label:
                        correct_predictions += 1
                    total_predictions += 1

                    if predicted_label.item() == true_label_1:
                        correct_predictions_1 += 1

                    if predicted_label.item() == true_label_2:
                        correct_predictions_2 += 1
                mean_sim_array[fold_idx, epoch] = (sum(cosine_value_list) / len(cosine_value_list))
                sitem_predicted_list = []
                sitem_predicted_label_list = []
                probs_epoch = []
                for sitem_id, sitem in enumerate(used_sub_item_list):
                    output_sitem = model(torch.tensor([tokenizer.encode(sitem)]).to(model.device), output_hidden_states=True)
                    sitem_label =  linear_net(output_sitem.hidden_states[-1][-1][-1])
                    p1 = F.softmax(sitem_label, dim=0)[1].item() 
                    _, sitem_predicted_label = torch.max(sitem_label, dim=0)
                    probs_epoch.append(p1)
                    sitem_predicted_list.append(sitem_predicted_label)
                yhat_prob = np.asarray(probs_epoch, dtype=float)
                for itemmm in sitem_predicted_list:
                    if itemmm.item() == 0:
                        sitem_predicted_label_list .append('true')
                    else:
                        sitem_predicted_label_list .append('false')
                sub_day7_predictions.append(sitem_predicted_label_list)
                test_acc = accuracy_score(sitem_predicted_label_list, sub_item_train)
                chonghe_acc =  accuracy_score(sub_seven_acc_zero_one,sitem_predicted_label_list)

                NP_acc = accuracy_score(sitem_predicted_label_list[:48], sub_item_train[:48])
                NPVP_acc = accuracy_score(sitem_predicted_label_list[48:96], sub_item_train[48:96])
                SOV_acc = accuracy_score(sitem_predicted_label_list[96:], sub_item_train[96:])
                #print(f'test acc : {test_acc}')
                test_acc_array[fold_idx, epoch] = test_acc
                chonghe_acc_array[fold_idx, epoch] = chonghe_acc
                
                np_acc_array[fold_idx, epoch] = NP_acc 
                npvp_acc_array[fold_idx, epoch] = NPVP_acc
                sov_acc_array[fold_idx, epoch] = SOV_acc
        



        
                cla_accuracy = correct_predictions / total_predictions if total_predictions else 0
                cla_accuracy_1 = correct_predictions_1 / total_predictions if total_predictions else 0
                cla_accuracy_2 = correct_predictions_2 / total_predictions if total_predictions else 0
                
                valid_acc_array[fold_idx, epoch] = max(cla_accuracy_1, cla_accuracy_2)
                valid_acc_array_true[fold_idx, epoch] = cla_accuracy
                if cla_accuracy > biggest_like_huamn:
                    biggest_like_huamn = cla_accuracy
                    torch.save({
                        'gpt2_model': model.state_dict(),
                        'linear_net': linear_net.state_dict()
                    }, os.path.join(best_model_path, 'best_model.pt'))

        print(biggest_like_huamn)  
        fold_idx += 1
        print("subject : ", subject, " fold ", fold_idx, " end ")


    
    torch.cuda.empty_cache()
    gc.collect()
    print(subject, 'end!')


    return (
        acc_array,
        similarity_array,
        present_array,
        train_acc_array,
        valid_acc_array,
        test_acc_array,
        np_acc_array,
        npvp_acc_array,
        sov_acc_array,
        mean_sim_array,
        new_rate_array, 
        like_human_acc_array, 
        sub_day7_predictions, 
        chonghe_acc_array,
        valid_acc_array_true
    )

    



In [ ]:
from joblib import Parallel, delayed

results = Parallel(n_jobs=5)(
    delayed(process_subject)(subject)
    for subject in subjects
)

# 在主进程中赋值
for idx, subject in enumerate(subjects):
    acc_array_list[idx] = results[idx][0]
    similarity_array_list[idx] = results[idx][1]
    present_array_list[idx] = results[idx][2]
    train_acc_list[idx] = results[idx][3]
    valid_acc_list[idx] = results[idx][4]
    test_acc_list[idx] = results[idx][5]
    np_acc_list[idx] = results[idx][6]
    npvp_acc_list[idx] = results[idx][7]
    sov_acc_list[idx] = results[idx][8]
    mean_sim_list[idx] = results[idx][9]
    gen_result[idx] = results[idx][10]
    like_human_acc_list[idx] = results[idx][11] 
    day7_predictions[idx].append(results[idx][12])
    chonghe_acc_list[idx] = results[idx][13]
    valid_acc_list_ture[idx] = results[idx][14] 


In [ ]:

import pickle

output_path = './results_cla_mask.pkl'  


results = {
    'acc_array_list': acc_array_list,
    'similarity_array_list': similarity_array_list,
    'present_array_list': present_array_list,
    'train_acc_list': train_acc_list,
    'valid_acc_list':valid_acc_list,
    'test_acc_list': test_acc_list,  
    'np_acc_list':np_acc_list,  
    'npvp_acc_list':npvp_acc_list,
    'sov_acc_list':sov_acc_list,
    'mean_sim_list':mean_sim_list,
    'gen_result':gen_result,
    'like_human_acc_list': like_human_acc_list, 
    'day7_predictions':day7_predictions, 
    'chonghe_acc_list': chonghe_acc_list,
    'valid_acc_list_ture':valid_acc_list_ture
}


with open(output_path, 'wb') as f:
    pickle.dump(results, f)

print(f"saved  {output_path}")
